# 08 — Entry into nonstop service: fixed-cost bounds and a merger counterfactual with entry

Following the course notes' two-stage entry game (Week 10): carriers first decide whether to offer nonstop service
on a route, then compete in prices. With demand and marginal costs estimated, the variable profit of each entry
decision can be computed, and observed decisions bound the fixed cost of nonstop service by revealed preference.

* **Incumbents.** A carrier offering nonstop service must earn at least the fixed cost from it:
  $F \le \Delta\Pi^{exit}$, the drop in the carrier's variable profit if it withdrew the nonstop product and all
  carriers re-priced (its own connecting product recaptures some passengers).
* **Potential entrants.** A carrier selling only connecting service on the route while operating at both
  endpoints chose not to add nonstop service: $F \ge \Delta\Pi^{entry}$, the gain from adding it.

The fixed cost is $F = \bar F(X) + \nu$, with $X$ = distance band × hub status. As in the notes, the model is
incomplete and the inequalities only partially identify $\bar F$. We assume $\nu$ is an expectational error, mean
zero given the carrier's information (Pakes, Porter, Ho and Ishii 2015), so averaging each inequality within a
cell gives bounds $E[\Delta\Pi^{entry}\mid X] \le \bar F(X) \le E[\Delta\Pi^{exit}\mid X]$. The notes' selection
problem (demand shocks known at entry) is assumed away, as the notes recommend in practice: the entrant's unobserved
quality is set to its expected value.

Demand is the nested logit with one air-travel nest (closed form, so products can be added and removed);
marginal costs come from its first-order conditions. Counterfactual nonstop products for potential entrants take
the carrier's connecting product in the market and add the estimated nonstop and distance effects on utility and on
log marginal cost.

**Counterfactual.** After a JetBlue–Spirit merger, are rival entry or the merged firm's withdrawal of duplicate
nonstop service likely, and how would entry change the predicted fare effects?

In [ ]:
%run 00_setup.ipynb

In [ ]:
%run 00_model_functions.ipynb

In [ ]:
IV = IV_SETS["baseline"]
est = load_json("demand_full_sample")
ALPHA, RHO = est["linear"]["nested_air"]["alpha"], est["linear"]["nested_air"]["rho"]
BETA = est["linear"]["nested_air"]["beta"]
p_all = pd.read_parquet(INTERIM / "products_airport_nondir_iv.parquet")
QS = {qlabel(*q) for q in QUARTERS[-4:]}
d = p_all[p_all.quarter.isin(QS)].sort_values(["market_ids", "carrier", "nonstop"]).reset_index(drop=True)
if SMOKE:
    d = d[d.route.isin(d.route.unique()[:400]) | d.carrier.isin(["B6", "NK"])].reset_index(drop=True)
print(f"alpha = {ALPHA:.3f}, rho = {RHO:.3f}; {d.market_ids.nunique():,} markets in {sorted(QS)}")

## Nested logit (single inside nest): shares, Jacobian, equilibrium

In [ ]:
def nl_shares(delta):
    e = np.exp((delta - delta.max()) / (1 - RHO))
    D = e.sum()
    sg_cond = e / D
    logD = np.log(D) + delta.max() / (1 - RHO)
    inc = (1 - RHO) * logD
    sg = 1 / (1 + np.exp(-inc))
    return sg_cond * sg, sg_cond

def nl_jac(delta):
    s, sc = nl_shares(delta)
    Jd = np.diag(s / (1 - RHO)) - (RHO / (1 - RHO)) * np.outer(s, sc) - np.outer(s, s)
    return ALPHA * Jd, s

def equilibrium(xi, c, firms, p0, tol=1e-10, maxit=3000):
    p = p0.copy()
    O = (firms[:, None] == firms[None, :]).astype(float)
    for _ in range(maxit):
        J, s = nl_jac(xi + ALPHA * p)
        p_new = c - np.linalg.solve(O * J.T, s)
        p_new = 0.5 * p + 0.5 * p_new
        if np.abs(p_new - p).max() < tol:
            p = p_new
            break
        p = p_new
    s, _ = nl_shares(xi + ALPHA * p)
    return p, s

def firm_profit(p, s, c, firms, f, M):
    sel = firms == f
    return float(((p - c) * s)[sel].sum() * M * 100)          # dollars per quarter

## Recover mean utilities net of price and marginal costs; cost and utility effects of nonstop service

In [ ]:
s0 = 1 - d.groupby("market_ids").shares.transform("sum")
sg = 1 - s0
d["xi"] = np.log(d.shares / s0) - RHO * np.log(d.shares / sg) - ALPHA * d.prices
pos = d.groupby("market_ids").indices
mc = np.empty(len(d))
for m, idx in pos.items():
    J, s = nl_jac(d.xi.values[idx] + ALPHA * d.prices.values[idx])
    firms = d.carrier.values[idx]
    O = (firms[:, None] == firms[None, :]).astype(float)
    mc[idx] = d.prices.values[idx] + np.linalg.solve(O * J.T, s)
d["mc"] = mc
assert np.allclose(nl_shares(d.xi.values[pos[next(iter(pos))]] + ALPHA * d.prices.values[pos[next(iter(pos))]])[0],
                   d.shares.values[pos[next(iter(pos))]])
ok = d.mc > 0
cost_fit = AbsorbingLS(np.log(d.mc[ok]), d.loc[ok, ["nonstop", "distance", "hub_presence"]],
                       absorb=pd.DataFrame({"r": d.route[ok].astype("category"), "c": d.carrier_fe[ok].astype("category"),
                                            "q": d.quarter[ok].astype("category")})).fit()
G = cost_fit.params
print("log mc effects:", G.round(4).to_dict(), "| utility effects:", {k: round(v, 3) for k, v in BETA.items()})

## Variable-profit gains from nonstop service

In [ ]:
d["max_presence"] = np.maximum(d.presence_a, d.presence_b)
rows = []
for m, idx in pos.items():
    xi, c, pr, firms = d.xi.values[idx], d.mc.values[idx], d.prices.values[idx], d.carrier.values[idx]
    M = d.market_size.values[idx[0]]
    s_obs = d.shares.values[idx]
    ns = d.nonstop.values[idx]
    base_profit = {f: firm_profit(pr, s_obs, c, firms, f, M) for f in set(firms)}
    for k, j in enumerate(idx):
        if ns[k] == 1:                       # incumbent nonstop: profit lost if withdrawn
            keep = np.arange(len(idx)) != k
            p1, s1 = equilibrium(xi[keep], c[keep], firms[keep], pr[keep])
            rows.append({"market_ids": m, "carrier": firms[k], "type": "incumbent", "route_distance": d.route_distance.values[j],
                         "max_presence": d.max_presence.values[j],
                         "dPi": base_profit[firms[k]] - firm_profit(p1, s1, c[keep], firms[keep], firms[k], M)})
    for k, j in enumerate(idx):              # potential entrant: connecting-only carrier present at both endpoints
        f = firms[k]
        if ns[k] == 0 and not ((firms == f) & (ns == 1)).any() and d.presence_a.values[j] > 0 and d.presence_b.values[j] > 0:
            ddist = d.route_distance.values[j] - d.distance.values[j]
            xi_new = xi[k] + BETA["nonstop"] + BETA["distance"] * ddist
            c_new = c[k] * np.exp(G["nonstop"] + G["distance"] * ddist)
            p1, s1 = equilibrium(np.r_[xi, xi_new], np.r_[c, c_new], np.r_[firms, f], np.r_[pr, c_new * 1.2])
            rows.append({"market_ids": m, "carrier": f, "type": "potential entrant", "route_distance": d.route_distance.values[j],
                         "max_presence": d.max_presence.values[j],
                         "dPi": firm_profit(p1, s1, np.r_[c, c_new], np.r_[firms, f], f, M) - base_profit[f]})
ent = pd.DataFrame(rows)
ent["distance band"] = pd.cut(ent.route_distance, [0, 0.5, 1.0, 1.5, 2.5, 10], labels=["<500", "500-1000", "1000-1500", "1500-2500", ">2500"])
ent["hub endpoint"] = np.where(ent.max_presence >= 0.25, "hub", "non-hub")
ent.to_parquet(INTERIM / "entry_profits.parquet", index=False)
ent.groupby("type").dPi.describe()

## Bounds on the quarterly fixed cost of nonstop service

In [ ]:
b = ent.groupby(["distance band", "hub endpoint", "type"], observed=True).dPi.agg(["mean", "size"]).unstack("type")
bounds = pd.DataFrame({
    "lower bound ($000/qtr)": b[("mean", "potential entrant")] / 1e3,
    "upper bound ($000/qtr)": b[("mean", "incumbent")] / 1e3,
    "potential entrants": b[("size", "potential entrant")], "incumbents": b[("size", "incumbent")]})
bounds["identified set non-empty"] = bounds["lower bound ($000/qtr)"] <= bounds["upper bound ($000/qtr)"]
save_table(bounds, "entry_bounds", float_format="%.1f",
           note="Moment-inequality bounds on the mean quarterly fixed cost of offering nonstop service. Lower bound: mean variable-profit "
                "gain from adding nonstop service among carriers that sell only connecting service on the route while operating at both "
                "endpoints. Upper bound: mean variable-profit loss from withdrawing nonstop service among incumbents. Nested logit with "
                f"one air-travel nest; {sorted(QS)[0]}--{sorted(QS)[-1]}. Hub endpoint: carrier operates at least 25\\% of departures at an endpoint.")
bounds

## Counterfactual: JetBlue–Spirit with entry and exit responses (2025Q2)

In [ ]:
LATEST = qlabel(*LAST_Q)
cell_bounds = bounds[["lower bound ($000/qtr)", "upper bound ($000/qtr)"]] * 1e3
def cell_of(dist, pres):
    band = pd.cut([dist], [0, 0.5, 1.0, 1.5, 2.5, 10], labels=["<500", "500-1000", "1000-1500", "1500-2500", ">2500"])[0]
    key = (band, "hub" if pres >= 0.25 else "non-hub")
    return cell_bounds.loc[key] if key in cell_bounds.index else pd.Series([np.nan, np.nan], index=cell_bounds.columns)

cf = []
for m, idx in pos.items():
    if not m.endswith(LATEST):
        continue
    firms0 = d.carrier.values[idx]
    if not {"B6", "NK"} <= set(firms0):
        continue
    xi, c, pr, ns = d.xi.values[idx], d.mc.values[idx], d.prices.values[idx], d.nonstop.values[idx]
    M = d.market_size.values[idx[0]]
    w_party = np.where(np.isin(firms0, ["B6", "NK"]), d.passengers.values[idx], 0.0)
    merged = np.where(np.isin(firms0, ["B6", "NK"]), "B6", firms0)
    p_post, s_post = equilibrium(xi, c, merged, pr)
    dp_noentry = np.average(100 * (p_post / pr - 1), weights=w_party)
    out = {"market_ids": m, "route": d.route.values[idx[0]], "dp_parties_no_entry": dp_noentry, "entry": "none",
           "dp_parties_with_entry": dp_noentry, "merged_drops_nonstop": "no"}
    best_gain, best = -np.inf, None
    for k, j in enumerate(idx):
        f = firms0[k]
        if f in ("B6", "NK") or ns[k] == 1 or ((firms0 == f) & (ns == 1)).any():
            continue
        if not (d.presence_a.values[j] > 0 and d.presence_b.values[j] > 0):
            continue
        ddist = d.route_distance.values[j] - d.distance.values[j]
        xi_new = xi[k] + BETA["nonstop"] + BETA["distance"] * ddist
        c_new = c[k] * np.exp(G["nonstop"] + G["distance"] * ddist)
        p1, s1 = equilibrium(np.r_[xi, xi_new], np.r_[c, c_new], np.r_[merged, f], np.r_[p_post, c_new * 1.2])
        gain = firm_profit(p1, s1, np.r_[c, c_new], np.r_[merged, f], f, M) - firm_profit(p_post, s_post, c, merged, f, M)
        lo, hi = cell_of(d.route_distance.values[j], d.max_presence.values[j])
        if gain > best_gain:
            best_gain, best = gain, (f, lo, hi, p1)
    if best is not None:
        f, lo, hi, p1 = best
        status = "certain" if best_gain > hi else ("possible" if best_gain > lo else "none")
        out.update({"entrant": f, "entry_gain": best_gain, "F_lower": lo, "F_upper": hi, "entry": status})
        if status != "none":
            out["dp_parties_with_entry"] = np.average(100 * (p1[:len(idx)] / pr - 1), weights=w_party)
    party_ns = [k for k in range(len(idx)) if firms0[k] in ("B6", "NK") and ns[k] == 1]
    if len(party_ns) == 2:                   # merged firm with two nonstop products: is withdrawing one profitable?
        losses = []
        for k in party_ns:
            keep = np.arange(len(idx)) != k
            p2, s2 = equilibrium(xi[keep], c[keep], merged[keep], p_post[keep])
            losses.append(firm_profit(p_post, s_post, c, merged, "B6", M) - firm_profit(p2, s2, c[keep], merged[keep], "B6", M))
        lo, hi = cell_of(d.route_distance.values[idx[0]], 1.0)
        out["merged_drops_nonstop"] = "likely" if min(losses) < lo else ("possible" if min(losses) < hi else "no")
    out["w"] = w_party.sum()
    cf.append(out)
cf = pd.DataFrame(cf)
cf.to_csv(RES / "b6nk_entry_counterfactual.csv", index=False)
summ = pd.DataFrame({
    "overlap markets": [len(cf)],
    "rival nonstop entry certain": [(cf.entry == "certain").sum()], "rival nonstop entry possible": [(cf.entry == "possible").sum()],
    "merged firm likely/possibly drops a duplicate nonstop": [(cf.merged_drops_nonstop != "no").sum()],
    "B6/NK fare change, no entry (%)": [np.average(cf.dp_parties_no_entry, weights=cf.w)],
    "B6/NK fare change, with certain entry (%)": [np.average(np.where(cf.entry == "certain", cf.dp_parties_with_entry, cf.dp_parties_no_entry), weights=cf.w)],
    "B6/NK fare change, with certain + possible entry (%)": [np.average(cf.dp_parties_with_entry, weights=cf.w)]}).T
summ.columns = ["NL (air travel)"]
save_table(summ, "b6nk_entry", float_format="%.2f",
           note="JetBlue--Spirit merger in 2025Q2 with entry responses. For each overlap market, the most profitable potential nonstop "
                "entrant's post-merger variable-profit gain is compared with the fixed-cost bounds of its cell: above the upper bound entry is "
                "certain, between the bounds possible. Fare changes are passenger-weighted over B6/NK products.")
summ

## Nonstop entry and exit in the data (T-100 confirmed products)

In [ ]:
ns_prod = p_all[p_all.nonstop == 1].assign(key=lambda x: x.route + ":" + x.carrier)
present = ns_prod.pivot_table(index="key", columns="t", values="passengers", aggfunc="sum").notna()
flows = pd.DataFrame({"entries": (present & ~present.shift(1, axis=1, fill_value=False)).sum().iloc[1:],
                      "exits": (~present & present.shift(1, axis=1, fill_value=False)).sum().iloc[1:],
                      "nonstop products": present.sum().iloc[1:]})
flows.index = [qlabel(*QUARTERS[t]) for t in flows.index]
save_table(flows, "nonstop_entry_exit", float_format="%.0f",
           note="Carrier-route nonstop products appearing (entries) or disappearing (exits) relative to the previous quarter.")
flows

In [ ]:
save_json({"bounds": bounds.reset_index().astype({"distance band": str}).to_dict("records"), "counterfactual": summ.to_dict(),
           "cost_effects": G.to_dict()}, "entry")